### 3B1B 特征值/特征向量 - 极简形象笔记

*   **脑海画面：** 空间被矩阵像面团一样拉扯。绝大多数箭头都被扯歪了，只有几根特殊的箭头死死钉在原来的直线上，仅仅是变长或变短了。
*   **角色分配：** 
    *   **特征向量：** 那些“方向不偏离、不转弯”的固执箭头。
    *   **特征值：** 这些固执箭头被“拉长/压缩”的倍数（比如变成原来的 2 倍，特征值就是 2；如果反向了，特征值就是负数）。
*   **终极奥义：** $A\mathbf{v} = \lambda\mathbf{v}$。找到它们，就可以把复杂的“矩阵空间变换”，简化成纯粹的“数字缩放”。

### 3B1B 抽象向量空间 - 极简形象笔记

*   **思想觉醒：** 撕掉向量“箭头”和“数字列表”的伪装。只要能“相加”和“缩放”，万物皆可为向量。
*   **降维打击实例：**
    *   **函数 = 向量：** $x^2$ 和 $\sin(x)$ 本质上和指向右上角的箭头没有区别，它们只是无限维空间里的一个点。
    *   **求导 = 矩阵：** 微积分里的求导操作，本质上就是一个矩阵在“函数向量空间”里做了一次空间变换。
*   **核心直觉：** 数学不是关于“箭头”的，而是关于“规则”的。只要遵守加法和数乘的规则，线性代数的所有结论（特征值、行列式、基变换）在任何抽象领域（如量子力学、微积分）都能无缝运行。

### 李宏毅机器学习：分类与损失函数 - 极简形象笔记

*   **分类的硬伤（回归 vs One-hot）：** 绝对不能用普通的数字（1、2、3）当分类标签，否则数学上会强行认为“1号类和2号类更近”。必须换成独热向量（One-hot Vector），让所有类别在空间里保持绝对的两两等距。
*   **Softmax 的两把刷子：** 模型的原始输出（Logits）是任意实数，无法直接和 0/1 标签对齐。Softmax 强行做了两件事：一是通过指数化（Exponential）“拉大贫富差距”（让得分高的更高，低的趋近于0）；二是通过归一化变成合为 1 的“概率分布”。
*   **优化地形对决（MSE vs 交叉熵）：** 当模型“极度差劲/完全猜错”时：
    *   **MSE：** 对应的是一片一望无际的**平坦红高原**。梯度（斜率）极其微弱，模型直接卡死在原地动弹不得。
    *   **交叉熵（Cross-Entropy）：** 对应的是一条**陡峭的下滑道**。错得越惨，斜率越大，给模型的动力越足，能迅速带模型起步走出困境。
*   **PyTorch 潜规则：** 交叉熵和 Softmax 是一对锁死的铁杆组合。在 PyTorch 中，`CrossEntropyLoss` 内部已经自动内建了 Softmax，模型架构的最后一层千万别自己重复叠加。

### 3.4.1 分类问题 - 极简形象笔记

*   **避坑指南：** 绝对不能用数字大小（1=狗，2=猫，3=鸡）代表类别，会给模型强加荒谬的“顺序”和“中间态”（猫不是狗和鸡的平均值）。
*   **核心武器（独热编码 One-Hot）：** 把类别变成“排灯”。3 个类别就是 3 个灯泡，轮到谁，谁对应的位置就亮 `1`，其余灭 `0`（例如：猫 = `[0, 1, 0]`）。
*   **架构影响：** 既然答案变成了 3 个灯泡，模型的输出端口也必须从 1 个裂变成 3 个，分别给出每个类别的“自信得分”。

### 3.4.2 网络架构 - 极简形象笔记

*   **脑海画面（全连接/Dense）：** 左边是输入阵营（特征），右边是输出阵营（类别得分）。左边每个士兵都要向右边每个长官汇报，交织成一张密集的“全连接网”。
*   **得分公式（矩阵乘法）：** 
    *   某个类别的得分 = 所有输入特征 $\times$ 对应的专属权重 + 这个类别的偏置（底分）。
    *   代码直觉：$\mathbf{o} = \mathbf{W}\mathbf{x} + \mathbf{b}$。不要用 for 循环算 3 次，用矩阵乘法一刀切算完。
*   **行规（层数判定）：** 输入层不算层！因为数据只是进去逛了一圈，只做了一次权重运算就直达输出了。没有“中间商”（隐藏层），所以叫**单层神经网络**。

### 3.4.3 全连接层的参数开销 - 极简形象笔记

*   **算账公式：** 参数量 $\approx$ 输入特征数 $d$ $\times$ 输出类别数 $q$（即 $\mathcal{O}(dq)$）。
*   **现实痛点（显卡杀手）：** 处理几十像素的小图没问题。但如果用来处理上千万像素的高清图，单层参数量直接破百亿，普通显卡瞬间显存爆炸（OOM）。
*   **核心伏笔：** 知道“全连接”很贵，是为了将来学“卷积神经网络（CNN）”做铺垫。CNN 就是为了解决这个“昂贵代价”而诞生的救星。本章图片小，我们先将就着用。

### 3.4.4 softmax运算 - 极简形象笔记

*   **脑海画面（流水线加工）：** 
    1. 各种乱七八糟的原始得分（有正有负）进入了 $e^x$ 烤箱。
    2. 烤出来全部变成了热腾腾的正数，而且原来分数高的膨胀得特别巨大（拉大差距）。
    3. 最后经过一把均分刀，全部除以总和，变成了加起来等于 1 的百分比概率。
*   **三大法宝（输出特性）：** 1. 全是非负数；2. 加起来等于 1；3. **绝不改变原有的排名顺序**。
*   **工程实操大实话：** 
    *   **真正预测时：** 直接找原始得分最大的，压根不用算 Softmax（省算力）。
    *   **为什么还要它：** 训练算 Loss 时必须要用概率；以及产品经理要你显示“模型有 99% 的把握”时必须要用。

### 3.4.5 小批量样本的矢量化 - 极简形象笔记

*   **痛点与破局：** 用 for 循环一张一张算图片会浪费 GPU 算力。必须把 n 张图片装进一个集装箱（Minibatch）一起处理。
*   **矩阵化身（形状变化）：** 
    *   特征不再是向量，而是变成了矩阵 X（形状：n 行 d 列，每一行就是一张图）。
    *   结果不再是向量，而是变成了矩阵 O（形状：n 行 q 列，每一行是一张图的各个类别得分）。
*   **一步到位公式：** O = XW + b。利用矩阵乘法，一次运算干完 n 张图的活，速度直接起飞。
*   **Softmax 流水线：** 对得分矩阵 O 永远是“按行处理”（Rowwise）。每张图片（每一行）管好自己的概率转换，互相不串门。

### 3.4.6 损失函数 - 极简形象笔记

*   **核心任务** 衡量“预测概率(y_hat)”和“真实标签(y)”之间的差距，有了差距才知道怎么惩罚模型。
*   **弃用 MSE，拥抱交叉熵** 回归问题用 MSE，分类问题必须用交叉熵 Loss（本质是求最大似然估计）。
*   **独热向量的“消行魔法”** 因为真实标签（y）是个只有 1 个 1、其他全是 0 的独热向量，所以交叉熵复杂的连加公式中，99% 的项都被 0 乘没了。
*   **只盯一点（核心化简）** 最终的 Loss 公式化简为 `-log(正确类别的预测概率)`。
    *   **直觉** 我们完全不管模型给“错的类别”瞎猜了多少概率，我们只死死抓着“正确类别”对应的那个概率。它越接近 1，Loss 越小；它越接近 0，Loss 越无穷大（给予最严厉的惩罚）。

### 3.4.7 信息论基础 - 极简形象笔记

*   **脑海画面（发报员的困境）：** 真理（P）是你要发报的客观现实，而模型（Q）是你手里那本主观编制的“电报密码本”。
*   **信息量（Surprise）：** 一件事越离谱（概率越低），发生时你就越震惊，包含的信息量就越大（公式 `-log P`）。
*   **熵（Entropy）：** 真理 P 固有的不确定性底线。也就是用“最完美的密码本”发报所需的最少流量。
*   **交叉熵（Cross-Entropy）：** 拿着“半吊子模型 Q 的密码本”，去发送“真理 P 的数据”，最终耗费的实际流量。
*   **终极奥义：** 密码本越烂，耗费的流量（交叉熵）就越大。我们训练模型拼命“最小化交叉熵”，本质就是在修补密码本，让模型 Q 完美对齐客观真理 P，不再浪费一丁点比特。

### 3.4.8 模型预测和评估 - 极简大白话笔记

*   **日常应用（预测 Prediction）：** 遇到事情不要慌，直接用 `argmax`。不用管具体的概率数字是多少，直接挑出得分最高的那一项作为最终定论。
*   **KPI 汇报（评估 Accuracy）：** 做对的题数 / 总题数 = 准确率（精度）。这是产品经理和用户唯一听得懂的人话指标。
*   **灵魂拷问：为啥只看准确率，训练却要用交叉熵 Loss？**
    *   **准确率是“死台阶”（不可导）：** 考 1 分和 59 分结果都是“挂科”。微小的进步无法体现，梯度为 0，模型不知道往哪走。
    *   **交叉熵是“平滑坡”（可导）：** 哪怕是从 1 分考到 2 分，交叉熵也能敏锐地捕捉到（Loss 降低），并给模型指明继续优化的平滑方向。**（用交叉熵训练，用准确率评估）**